# Tokenizer Examples 2, Long Context and Context Splitting

In this Jupyter notebook, we explore tokenization in the context of QUESTION-ANSWER language modeling. In this type of modeling, both the question and the context — which provides the basis for the answer — are fed to the language model simultaneously. Consequently, both the question and the context need to be tokenized together. Since the combined length of the question and context may exceed the maximum input length that the language model can handle, the input must be divided into smaller, manageable chunks. This process is accomplished using a technique known as the sliding window, which ensures that the entire answer to the question is contained within one of the chunks. It is important to avoid a scenario where the beginning of the answer is in one chunk and the end is in another.

Figure 1. illustrates how the sliding window (red) operates. The long sentence (blue) contains both the question and the context. Since the tokenized version is too long to be processed by the LLM directly, we need to break down the sentence into smaller chunks. If we were to simply divide the long text into smaller chunks, as shown in green, the answer to the question might be split across multiple context "windows." By using a sliding window with a specified stride, we reduce the likelihood that the answer is split between two sub-sentences.

and the answer-containing context will end up in separate chunks.

<figure align="center">
    <img src="./images/tokenization-long-context.png" width="500">
    <figcaption>Figure 1. Sliding window over long sentences.</figcaption>
</figure>

When tokenizing long texts, or tokenizing several inputs, the output from the tokenizer has the following important functions / variables:
* `overflow_to_sample_mapping` attribute
  * enabled if `return_overflowing_tokens = True`
  * Example: `[0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2]`
  * Maps input phrases, given in a list, to the context windows. In the example above, the first sentence is broken into 0 context windows etc.
* `offset_mapping` attribute
  * enabled if `return_offset_mapping = True`
  * Maps token positions to character locations in the original text.
  * Example: 
    * The text `How long is a piece of string?` is tokenized to `[101, 1731, 1263, 1110, 170, 2727, 1104, 170, 5101, 136, 102]`
    * The tokenized version, when decoded is `[CLS] How long is a piece of a string? [SEP]`, so `CLS` and `SEP` have been added. These correspond to id:s 101 and 102
    * Offset mapping is: `[(0, 0), (0, 3), (4, 8), (9, 11), (12, 13), (14, 19), (20, 22), (23, 24), (25, 31), (31, 32), (0, 0)]`
    * `(0, 3)` means that the word, related to token 1, starts at position 0 and ends at position 3 in the string. This corresponds to `How `
* `sequence_ids(n)` function
  * Example: `[None, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, None, 1, 1, ..., None]`
  * Special tokens, like `CLS` and `SEP`, correspond to `None`
  * 0:s correspond to the tokens of the first sentence (question)
  * 1:s correspond to the tokens of the second sentence (context)

In [21]:
from datasets import load_dataset
from transformers import AutoTokenizer
from pprint import pp
from typing import List, Tuple

In [22]:
# This is the model we will be using
model_checkpoint = "distilbert-base-cased"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [23]:
# Let's load the SQuAD (Stanford Question Answering Dataset) dataset.
raw_dataset = load_dataset('squad')

Using the latest cached version of the dataset since squad couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at /home/jarno/.cache/huggingface/datasets/squad/plain_text/0.0.0/7b6d24c440a36b6815f21b70d25016731768db1f (last modified on Fri Aug  9 15:42:09 2024).


In [24]:
# Take a glimpse at the dataset
print(raw_dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [25]:
# Let's take a glimpse at the first training example
pp(raw_dataset["train"][0])

{'id': '5733be284776f41900661182',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the '
            "Main Building's gold dome is a golden statue of the Virgin Mary. "
            'Immediately in front of the Main Building and facing it, is a '
            'copper statue of Christ with arms upraised with the legend '
            '"Venite Ad Me Omnes". Next to the Main Building is the Basilica '
            'of the Sacred Heart. Immediately behind the basilica is the '
            'Grotto, a Marian place of prayer and reflection. It is a replica '
            'of the grotto at Lourdes, France where the Virgin Mary reputedly '
            'appeared to Saint Bernadette Soubirous in 1858. At the end of the '
            'main drive (and in a direct line that connects through 3 statues '
            'and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858

In [26]:
# In the training data there should be only one answer, 
# but we can make sure by filtering out training examples that have more than 1 or no answers at all
raw_dataset["train"].filter(lambda x: len(x["answers"]["text"]) != 1)

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 0
})

In [27]:
# Let's take a look at the validation answers and questions
val_answers = raw_dataset["validation"][:]["answers"]
val_questions = raw_dataset["validation"][:]["question"]

# Indices to answers in the validation data set that contain only 1 answer
ind_single_answer = [ind for ind, elem in enumerate(val_answers) if len(elem["text"]) == 1]

# Indices to answers in the validation dataset that contain more than 1 answer
ind_many_answers = [ind for ind, elem in enumerate(val_answers) if len(elem["text"]) > 1]

# Here we should have only 1 answer
print("-------------")
print("SINGLE ANSWER")
print("-------------")
print("CONTEXT")
pp(raw_dataset['validation'][ind_single_answer[0]]['context'])
print(f"Question: {raw_dataset['validation'][ind_single_answer[0]]['question']}")
print(f"Answers: {raw_dataset['validation'][ind_single_answer[0]]['answers']}")

# Here we should have many answers
print("")
print("------------")
print("MANY ANSWERS")
print("------------")
print("CONTEXT")
pp(raw_dataset['validation'][ind_many_answers[0]]['context'])
print(f"Question: {raw_dataset['validation'][ind_many_answers[0]]['question']}")
print(f"Answers: {raw_dataset['validation'][ind_many_answers[0]]['answers']}")

-------------
SINGLE ANSWER
-------------
CONTEXT
('On 24 March 1879, Tesla was returned to Gospić under police guard for not '
 'having a residence permit. On 17 April 1879, Milutin Tesla died at the age '
 'of 60 after contracting an unspecified illness (although some sources say '
 'that he died of a stroke). During that year, Tesla taught a large class of '
 'students in his old school, Higher Real Gymnasium, in Gospić.')
Question: What do some sources say Nikola's brother died from?
Answers: {'text': ['a stroke'], 'answer_start': [239]}

------------
MANY ANSWERS
------------
CONTEXT
('Super Bowl 50 was an American football game to determine the champion of the '
 'National Football League (NFL) for the 2015 season. The American Football '
 'Conference (AFC) champion Denver Broncos defeated the National Football '
 'Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super '
 "Bowl title. The game was played on February 7, 2016, at Levi's Stadium in "
 'the San F

In [28]:
# Verify what is the maximum length of the tokenizer
tokenizer_max_length = tokenizer.model_max_length
print(f"Tokenizer maximum length: {tokenizer_max_length}")

Tokenizer maximum length: 512


In [29]:
# Here we verify how the combination of QUESTION - CONTEXT are tokenized using a sliding window. Since the context
# can be much longer that the maximum number of allowed tokens, we use a sliding window to form a number of
# tokenized QUESTION - CONTEXT id:s. The QUESTION id:s are added to the beginning of each set of id:s
context = "String refers to a thin, flexible piece of material that is long and narrow, often made of fibers twisted \
together to form a strong, durable thread. Strings can be made from various materials, including natural fibers \
like cotton, wool, or silk, as well as synthetic fibers such as nylon or polyester.The structure of a string allows \
it to be both strong and flexible, making it suitable for a wide range of applications. Strings are commonly used in \
sewing, crafting, and binding, as well as in musical instruments where they are stretched tightly to produce sound. \
In addition, strings can also serve functional purposes, such as tying objects together or being woven into larger \
fabrics and textiles. Their versatility and adaptability make them an essential component in many everyday items \
and industrial processes."

question = "How long is a piece of a string?"

print('CONTEXT')
pp(context)
print("")
print('QUESTION')
pp(question)

# Tokenize without truncation first to see the combined length
tokenized_without_truncation = tokenizer(question, context, truncation=False)
print("")
print("Total length without truncation:", len(tokenized_without_truncation['input_ids']))

# Tokenized question
question_tokenized = tokenizer(question)

# Question - Context, tokenized
inputs = tokenizer(question, context, max_length=30, truncation="only_second", stride=10, return_overflowing_tokens=True, return_offsets_mapping=True)

CONTEXT
('String refers to a thin, flexible piece of material that is long and narrow, '
 'often made of fibers twisted together to form a strong, durable thread. '
 'Strings can be made from various materials, including natural fibers like '
 'cotton, wool, or silk, as well as synthetic fibers such as nylon or '
 'polyester.The structure of a string allows it to be both strong and '
 'flexible, making it suitable for a wide range of applications. Strings are '
 'commonly used in sewing, crafting, and binding, as well as in musical '
 'instruments where they are stretched tightly to produce sound. In addition, '
 'strings can also serve functional purposes, such as tying objects together '
 'or being woven into larger fabrics and textiles. Their versatility and '
 'adaptability make them an essential component in many everyday items and '
 'industrial processes.')

QUESTION
'How long is a piece of a string?'

Total length without truncation: 172


In [30]:
# Each of the tokenized id:s begin with the question added to the beginning, followed by the context.
# Since long contexts are broken into smaller chunks, the overflow_to_sample_mapping contains the mapping from
# from the list of id:s to contexts. For example, [0, 0] means that the first context has been broken into 2 set of input id:s.
# Therefore
# - inputs['input_ids'][0]
# - inputs['input_ids'][1]
# map to the first QUESTION - CONTEXT
print(f"question, tokenized: {question_tokenized['input_ids']}")
print(f"inputs 0: {inputs['input_ids'][0]}")
print(f"inputs 1: {inputs['input_ids'][1]}")
print(f"overflow_to_sample_mapping: {inputs['overflow_to_sample_mapping']}")

print("")
print(f"inputs 0, decoded: {tokenizer.decode(inputs['input_ids'][0])}")
print(f"inputs 1, decoded: {tokenizer.decode(inputs['input_ids'][1])}")

question, tokenized: [101, 1731, 1263, 1110, 170, 2727, 1104, 170, 5101, 136, 102]
inputs 0: [101, 1731, 1263, 1110, 170, 2727, 1104, 170, 5101, 136, 102, 13604, 4431, 1106, 170, 4240, 117, 13156, 2727, 1104, 2578, 1115, 1110, 1263, 1105, 4142, 117, 1510, 1189, 102]
inputs 1: [101, 1731, 1263, 1110, 170, 2727, 1104, 170, 5101, 136, 102, 1104, 2578, 1115, 1110, 1263, 1105, 4142, 117, 1510, 1189, 1104, 18064, 6061, 1487, 1106, 1532, 170, 2012, 102]
overflow_to_sample_mapping: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

inputs 0, decoded: [CLS] How long is a piece of a string? [SEP] String refers to a thin, flexible piece of material that is long and narrow, often made [SEP]
inputs 1, decoded: [CLS] How long is a piece of a string? [SEP] of material that is long and narrow, often made of fibers twisted together to form a strong [SEP]


In [31]:
# The offset_mapping contains a list of tuples with the start and end position of each of the tokens/words.
start_token = inputs['offset_mapping'][0][0]
word1_token = inputs['offset_mapping'][0][1]
word2_token = inputs['offset_mapping'][0][2]
print(f"offset_mapping: {inputs['offset_mapping']}")
print(f"start_token: {start_token}")
print(f"word1_token: {word1_token}")
print(f"word2_token: {word2_token}")

print("")
print("Offset_mapping to input word mapping:")
print(f"{word1_token} : {question[word1_token[0]:word1_token[1]]}")
print(f"{word2_token} : {question[word2_token[0]:word2_token[1]]}")

offset_mapping: [[(0, 0), (0, 3), (4, 8), (9, 11), (12, 13), (14, 19), (20, 22), (23, 24), (25, 31), (31, 32), (0, 0), (0, 6), (7, 13), (14, 16), (17, 18), (19, 23), (23, 24), (25, 33), (34, 39), (40, 42), (43, 51), (52, 56), (57, 59), (60, 64), (65, 68), (69, 75), (75, 76), (77, 82), (83, 87), (0, 0)], [(0, 0), (0, 3), (4, 8), (9, 11), (12, 13), (14, 19), (20, 22), (23, 24), (25, 31), (31, 32), (0, 0), (40, 42), (43, 51), (52, 56), (57, 59), (60, 64), (65, 68), (69, 75), (75, 76), (77, 82), (83, 87), (88, 90), (91, 97), (98, 105), (106, 114), (115, 117), (118, 122), (123, 124), (125, 131), (0, 0)], [(0, 0), (0, 3), (4, 8), (9, 11), (12, 13), (14, 19), (20, 22), (23, 24), (25, 31), (31, 32), (0, 0), (77, 82), (83, 87), (88, 90), (91, 97), (98, 105), (106, 114), (115, 117), (118, 122), (123, 124), (125, 131), (131, 132), (133, 135), (135, 140), (141, 147), (147, 148), (149, 156), (157, 160), (161, 163), (0, 0)], [(0, 0), (0, 3), (4, 8), (9, 11), (12, 13), (14, 19), (20, 22), (23, 24), (

In [32]:
# Let's tokenize first 3 samples from the raw validation dataset
inputs = tokenizer(
    raw_dataset["train"][:3]["question"],
    raw_dataset["train"][:3]["context"],
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
    return_offsets_mapping=True)

print(f"overflow_to_sample_mapping: {inputs['overflow_to_sample_mapping']}")
print("")

for ids in inputs['input_ids']:
    print(tokenizer.decode(ids))

overflow_to_sample_mapping: [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2]

[CLS] To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? [SEP] Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend " Venite Ad Me Omnes ". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basi [SEP]
[CLS] To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? [SEP] the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend " Venite Ad Me Omnes ". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin [SEP]
[CLS] To whom did the Virgin Mary allegedly appear i

In [33]:
# Here val points to the actual text -> raw_dataset['train'][val], 
# and i points to the correspoinding offset_mapping
for i, val in enumerate(inputs['overflow_to_sample_mapping']):
    if val == 0:
        mappings = inputs['offset_mapping'][i]
        # Choose between 'question' and 'context'
        cntr = -1
        text_mapping = {0: "question", 1: "context"}

        for mapping in mappings:
            start = mapping[0]
            end = mapping[1]
            if mapping == (0, 0):
                cntr += 1
            else:
                text = raw_dataset['train'][val][text_mapping[cntr]]
                print(f"{mapping} : {text[start:end]}")

(0, 2) : To
(3, 7) : whom
(8, 11) : did
(12, 15) : the
(16, 22) : Virgin
(23, 27) : Mary
(28, 37) : allegedly
(38, 44) : appear
(45, 47) : in
(48, 52) : 1858
(53, 55) : in
(56, 59) : Lou
(59, 63) : rdes
(64, 70) : France
(70, 71) : ?
(0, 13) : Architectural
(13, 15) : ly
(15, 16) : ,
(17, 20) : the
(21, 27) : school
(28, 31) : has
(32, 33) : a
(34, 42) : Catholic
(43, 52) : character
(52, 53) : .
(54, 56) : At
(56, 58) : op
(59, 62) : the
(63, 67) : Main
(68, 76) : Building
(76, 77) : '
(77, 78) : s
(79, 83) : gold
(84, 88) : dome
(89, 91) : is
(92, 93) : a
(94, 100) : golden
(101, 107) : statue
(108, 110) : of
(111, 114) : the
(115, 121) : Virgin
(122, 126) : Mary
(126, 127) : .
(128, 139) : Immediately
(140, 142) : in
(143, 148) : front
(149, 151) : of
(152, 155) : the
(156, 160) : Main
(161, 169) : Building
(170, 173) : and
(174, 180) : facing
(181, 183) : it
(183, 184) : ,
(185, 187) : is
(188, 189) : a
(190, 196) : copper
(197, 203) : statue
(204, 206) : of
(207, 213) : Christ
(21

In [34]:
# Special tokens, like CLS and SEP, correspond to None
# [None, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, None, 1, 1, ..., None]
# 0:s correspond to the tokens of the first sentence (question)
# 1:s correspond to the tokens of the second sentence (context)
print(inputs.sequence_ids(0))

[None, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, None, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, None]


In [35]:
first_occurence = min(i for i, val in enumerate(inputs.sequence_ids(0)) if val == 1)
last_occurence = max(i for i, val in enumerate(inputs.sequence_ids(0)) if val == 1)
print(f"Sequence length: {len(inputs.sequence_ids(0))}")
print(f"First occurrence of 1 at location: {first_occurence}")
print(f"Last occurrence of 1 at location: {last_occurence}")

Sequence length: 100
First occurrence of 1 at location: 17
Last occurrence of 1 at location: 98


## Mapping Text to the Context Window

In this section we map in which context window(s) the answer to the question lies in

In [36]:
def find_sequence_in_list(lst: List[int], seq: List[int]) -> Tuple[int, int]:
    """Checks if the sequence of integers defined by 'seq' exists in the 'lst'.

    Example:
    start_index, end_index = find_sequence_in_list([1, 3, 5, 7, 9, 11, 13, 15], [7, 9, 11])

    Parameters
    ----------
    lst : List[int]
        List of integers where the sequence is searched
    seq : List[int]
        List of integers that is searched in 'lst'

    Returns
    -------
    Tuple[int, int]
        - Start index
        - End index

    Raises
    ------
    RuntimeError
        Raises RuntimeError is 'seq' is longer than 'lst'.
    """
    if len(seq) > len(lst):
        raise RuntimeError("Length of lst must by the same or higher than that of seq")
    
    seq_len = len(seq)
    for i in range(len(lst) - seq_len + 1):
        if lst[i:i+seq_len] == seq:
            return i, i + seq_len - 1  # return the first and last index of the sequence
    return -1, -1  # return (-1, -1) if the sequence is not found


def find_first_and_last_occurrence(lst: List, item) -> Tuple[int, int]:
    """Finds first and last occurence of an item in a list.

    Example:
    first_index, last_index = find_fist_and_last_occurence([None, 0, 0, 0, None, 1, 1], 1)
    first_index = 5
    last_index = 6

    Parameters
    ----------
    lst : List
        List of items
    item : _type_
        Item that is being searched for

    Returns
    -------
    Tuple[int, int]
        - Index to first occurrence of item in lst
        - Index to last occurrence of item in lst
    """
    first_occurence = min(i for i, val in enumerate(lst) if val == item)
    last_occurence = max(i for i, val in enumerate(lst) if val == item)
    return first_occurence, last_occurence

In [37]:
# Let's tokenize first 3 samples from the raw validation dataset
inputs = tokenizer(
    raw_dataset["train"][0]["question"],
    raw_dataset["train"][0]["context"],
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
    return_offsets_mapping=True)

answer_ids = tokenizer(raw_dataset["train"][0]["answers"]['text'][0])['input_ids'][1:-1]
print(f"Answer tokens: {answer_ids}")

Answer tokens: [2216, 17666, 6397, 3786, 1573, 25422, 13149]


In [38]:
context = raw_dataset["train"][0]["context"]
pp(context)
print(raw_dataset["train"][0]["answers"])

('Architecturally, the school has a Catholic character. Atop the Main '
 "Building's gold dome is a golden statue of the Virgin Mary. Immediately in "
 'front of the Main Building and facing it, is a copper statue of Christ with '
 'arms upraised with the legend "Venite Ad Me Omnes". Next to the Main '
 'Building is the Basilica of the Sacred Heart. Immediately behind the '
 'basilica is the Grotto, a Marian place of prayer and reflection. It is a '
 'replica of the grotto at Lourdes, France where the Virgin Mary reputedly '
 'appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive '
 '(and in a direct line that connects through 3 statues and the Gold Dome), is '
 'a simple, modern stone statue of Mary.')
{'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


In [40]:
# This code shows one way of finding the context windows that contain the answer
# This code is too slow for training purposes
for i, (input, token_word_mapping) in enumerate(zip(inputs['input_ids'], inputs['offset_mapping'])):
    # Indices to the tokens where the answer begins and ends in the question+context window
    answer_start_index, answer_end_index = find_sequence_in_list(input, answer_ids)

    if answer_start_index != -1:
        print(f"answer_start_index: {answer_start_index}, answer_end_index: {answer_end_index}")        
        print(f"Answer word mapping, start: {token_word_mapping[answer_start_index]}")
        print(f"Answer word mapping, end: {token_word_mapping[answer_end_index]}")
        
        answer_start_letter_location = token_word_mapping[answer_start_index][0]
        answer_end_letter_location = token_word_mapping[answer_end_index][1]

        print(f"Answer: {context[answer_start_letter_location:answer_end_letter_location]}")
        print("-------------------")

answer_start_index: 72, answer_end_index: 78
Answer word mapping, start: (515, 520)
Answer word mapping, end: (537, 541)
Answer: Saint Bernadette Soubirous
-------------------
answer_start_index: 40, answer_end_index: 46
Answer word mapping, start: (515, 520)
Answer word mapping, end: (537, 541)
Answer: Saint Bernadette Soubirous
-------------------


In [48]:
answer_start = raw_dataset["train"][0]["answers"]["answer_start"][0]
answer_end = answer_start + len(raw_dataset["train"][0]["answers"]["text"][0])

print(f"Answer start: {answer_start}")
print(f"Answer end: {answer_end}")

for i, (input, token_word_mapping) in enumerate(zip(inputs['input_ids'], inputs['offset_mapping'])):
    # Indices to where the context starts and ends in the question+context window
    start_context, end_context = find_first_and_last_occurrence(inputs.sequence_ids(i), 1)


Answer start: 515
Answer end: 541
